# راهنمای کاربردی Matplotlib و Seaborn

این نوت‌بوک یک **cheat sheet قابل اجرا** برای مهم‌ترین دستورهای رسم نمودار در Python است. توضیح‌ها فارسی و نام توابع و مثال‌ها انگلیسی‌اند تا بتوانی به‌راحتی در پروژه‌های واقعی از آن‌ها استفاده کنی.

## فهرست
1. نصب و آماده‌سازی
2. آماده‌سازی دادهٔ نمونه
3. مفاهیم پایهٔ Matplotlib
4. نمودارهای پرکاربرد Matplotlib
5. عنوان، محورها، راهنما، برچسب و ذخیره‌سازی
6. نمودارهای رابطه‌ای و توزیعی در Seaborn
7. نمودارهای دسته‌ای (Categorical)
8. رگرسیون و همبستگی
9. ماتریس همبستگی، pairplot و نمودارهای چندبخشی
10. رنگ‌ها، ظاهر نمودار و انتخاب نمودار مناسب

> تمام داده‌ها در همین نوت‌بوک به‌صورت مصنوعی ساخته می‌شوند؛ بنابراین برای اجرای مثال‌ها به دانلود فایل یا دسترسی اینترنت نیاز نیست. خروجی نمودارها نمونه‌اند و نتیجهٔ پژوهش واقعی محسوب نمی‌شوند.


## 1) نصب و وارد کردن کتابخانه‌ها
اگر کتابخانه‌ها نصب نیستند، دستور زیر را در ترمینال یا یک سلول Jupyter اجرا کن.

In [ ]:
# در صورت نیاز، این خط را در محیط خود اجرا کن:
# %pip install numpy pandas matplotlib seaborn jupyter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)

# ظاهر پیش‌فرض مناسب برای نمودارها
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 110


## 2) ساخت دادهٔ نمونه
این دادهٔ ساختگی برای مثال‌های نموداری استفاده می‌شود. در پروژهٔ خودت می‌توانی `df` را با دادهٔ واقعی و `df_sales` را با دیتافریم خودت جایگزین کنی.

In [ ]:
rng = np.random.default_rng(42)
n = 400

categories = np.array(["Electronics", "Home", "Clothing", "Sports"])
cities = np.array(["Tehran", "Shiraz", "Tabriz", "Isfahan", "Mashhad"])

category = rng.choice(categories, n, p=[0.30, 0.25, 0.25, 0.20])
city = rng.choice(cities, n)
unit_price = rng.normal(180, 55, n).clip(15, None)
quantity = rng.integers(1, 12, n)
discount = rng.uniform(0, 0.30, n)
revenue = unit_price * quantity * (1 - discount)
ad_spend = rng.uniform(5, 100, n)
profit = revenue * rng.uniform(0.06, 0.28, n) - ad_spend * 0.15
rating = rng.normal(3.8, 0.7, n).clip(1, 5)
age = rng.integers(18, 66, n)
months = pd.date_range("2025-01-01", periods=12, freq="MS")
date = rng.choice(months, n)

# افزودن کمی اثر دسته و شهر برای داشتن الگوهای قابل مشاهده
category_bonus = pd.Series(category).map({"Electronics": 50, "Home": 20, "Clothing": -10, "Sports": 5}).to_numpy()
city_bonus = pd.Series(city).map({"Tehran": 30, "Shiraz": 10, "Tabriz": 0, "Isfahan": 15, "Mashhad": -5}).to_numpy()
revenue = revenue + category_bonus + city_bonus
profit = revenue * rng.uniform(0.06, 0.25, n) - ad_spend * 0.15

df = pd.DataFrame({
    "date": date,
    "category": category,
    "city": city,
    "unit_price": unit_price.round(2),
    "quantity": quantity,
    "discount": discount.round(3),
    "revenue": revenue.round(2),
    "profit": profit.round(2),
    "ad_spend": ad_spend.round(2),
    "rating": rating.round(2),
    "age": age,
})

df_sales = df.groupby(["date", "category"], as_index=False).agg(
    revenue=("revenue", "sum"), profit=("profit", "sum"), quantity=("quantity", "sum")
)

print(df.shape)
display(df.head())


## 3) مفاهیم پایهٔ Matplotlib
- `plt.figure()` شکل جدید می‌سازد.
- `plt.subplots()` شکل و محورهای (Axes) را برمی‌گرداند؛ در پروژه‌های جدی این روش معمولاً انعطاف‌پذیرتر است.
- `fig` کل شکل است و `ax` یک محور/ناحیهٔ رسم است.

In [ ]:
# روش ساده با pyplot
plt.figure(figsize=(7, 4))
plt.plot(df_sales.groupby("date")["revenue"].sum().index,
         df_sales.groupby("date")["revenue"].sum().values,
         marker="o", linewidth=2, label="Revenue")
plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# روش Object-Oriented (پیشنهادی برای نمودارهای چندگانه)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
monthly = df_sales.groupby("date", as_index=False)["revenue"].sum()
ax.plot(monthly["date"], monthly["revenue"], marker="o", color="tab:blue")
ax.set(title="Monthly Revenue", xlabel="Month", ylabel="Revenue")
ax.tick_params(axis="x", labelrotation=45)
plt.show()


## 4) نمودارهای اصلی Matplotlib

In [ ]:
# 4.1 Line plot — روند در طول زمان
monthly = df_sales.groupby("date", as_index=False)["revenue"].sum()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(monthly["date"], monthly["revenue"], color="tab:blue", marker="o",
        linestyle="-", linewidth=2, markersize=5, label="Revenue")
ax.set(title="Line Plot", xlabel="Date", ylabel="Revenue")
ax.legend()
ax.grid(True, alpha=0.3)
ax.tick_params(axis="x", labelrotation=45)
plt.tight_layout(); plt.show()

# 4.2 Scatter plot — رابطهٔ دو متغیر
fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter(df["ad_spend"], df["revenue"], c=df["rating"], cmap="viridis",
                s=df["quantity"] * 12, alpha=0.7, edgecolors="white", linewidths=0.3)
ax.set(title="Ad Spend vs Revenue", xlabel="Ad Spend", ylabel="Revenue")
fig.colorbar(sc, ax=ax, label="Rating")
plt.tight_layout(); plt.show()

# 4.3 Bar chart و barh — مقایسهٔ دسته‌ها
by_category = df.groupby("category")["revenue"].mean().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axes[0].bar(by_category.index, by_category.values, color="steelblue")
axes[0].set(title="Vertical Bar", xlabel="Category", ylabel="Mean Revenue")
axes[0].tick_params(axis="x", labelrotation=25)
axes[1].barh(by_category.index, by_category.values, color="darkseagreen")
axes[1].set(title="Horizontal Bar", xlabel="Mean Revenue", ylabel="Category")
plt.show()

# 4.4 Histogram — توزیع متغیر عددی
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["revenue"], bins=25, color="cornflowerblue", edgecolor="white", alpha=0.9)
ax.set(title="Revenue Distribution", xlabel="Revenue", ylabel="Frequency")
plt.tight_layout(); plt.show()

# 4.5 Box plot — میانه، چارک‌ها و نقاط پرت
fig, ax = plt.subplots(figsize=(7, 4))
data = [df.loc[df["category"] == c, "revenue"] for c in categories]
ax.boxplot(data, tick_labels=categories, patch_artist=True, showmeans=True)
ax.set(title="Revenue by Category", xlabel="Category", ylabel="Revenue")
ax.tick_params(axis="x", labelrotation=20)
plt.tight_layout(); plt.show()


## 5) تنظیمات مهم Matplotlib
این دستورات را می‌توانی به تقریباً هر نموداری اضافه کنی.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df["ad_spend"], df["revenue"], alpha=0.55, color="tab:purple")

# عنوان و نام محورها
ax.set_title("Revenue vs Advertising Spend", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Advertising Spend", fontsize=11)
ax.set_ylabel("Revenue", fontsize=11)

# محدوده و تیک‌های محور
ax.set_xlim(0, 110)
ax.set_ylim(0, df["revenue"].quantile(0.99) * 1.08)
ax.set_xticks([0, 25, 50, 75, 100])
ax.tick_params(axis="both", labelsize=9, direction="out")

# شبکه، خطوط مرجع و توضیح روی نمودار
ax.grid(True, linestyle="--", alpha=0.25)
ax.axhline(df["revenue"].median(), color="red", linestyle="--", label="Median revenue")
ax.axvline(df["ad_spend"].mean(), color="green", linestyle=":", label="Mean ad spend")
ax.annotate("Example point", xy=(df["ad_spend"].iloc[0], df["revenue"].iloc[0]),
            xytext=(15, 20), textcoords="offset points", arrowprops={"arrowstyle": "->"})
ax.legend(loc="best", frameon=True)

# حذف حاشیهٔ اضافه و نمایش
fig.tight_layout()
plt.show()


In [ ]:
# رنگ، اندازه، شفافیت و نوع خط — تنظیمات متداول
# color / c        : رنگ
# alpha            : شفافیت از 0 تا 1
# linewidth / lw   : ضخامت خط
# linestyle / ls   : '-', '--', ':', '-.'
# marker           : 'o', 's', '^', 'x', '*', '.'
# markersize / ms  : اندازهٔ نشانگر
# figsize          : (عرض، ارتفاع) بر حسب اینچ
# dpi              : وضوح شکل

x = np.arange(1, 11)
y = x ** 1.5
fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
ax.plot(x, y, color="#E76F51", linewidth=2.5, linestyle="--", marker="D", markersize=5)
ax.fill_between(x, y * 0.85, y * 1.15, color="#E76F51", alpha=0.18, label="Illustrative band")
ax.set(title="Line Formatting Example", xlabel="X", ylabel="Y")
ax.legend()
plt.show()


### چند دستور مفید دیگر در Matplotlib

In [ ]:
# رسم میلهٔ خطا
means = df.groupby("category")["rating"].mean().reindex(categories)
stds = df.groupby("category")["rating"].std().reindex(categories)
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(categories, means, yerr=stds, fmt="o", capsize=4, color="black")
ax.set(title="Mean Rating with Standard Deviation", xlabel="Category", ylabel="Rating")
plt.show()

# نمودار دایره‌ای (برای تعداد کمی از دسته‌ها؛ اغلب bar chart خواناتر است)
counts = df["category"].value_counts()
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(counts.values, labels=counts.index, autopct="%.1f%%", startangle=90,
       counterclock=False, wedgeprops={"edgecolor": "white"})
ax.set_title("Category Share")
plt.show()

# چند نمودار در یک شکل
fig, axes = plt.subplots(2, 2, figsize=(10, 7), layout="constrained")
axes[0, 0].hist(df["revenue"], bins=20, color="steelblue")
axes[0, 0].set_title("Histogram")
axes[0, 1].scatter(df["ad_spend"], df["profit"], alpha=0.5)
axes[0, 1].set_title("Scatter")
axes[1, 0].bar(counts.index, counts.values, color="darkseagreen")
axes[1, 0].set_title("Counts")
axes[1, 0].tick_params(axis="x", labelrotation=20)
axes[1, 1].boxplot([df.loc[df.category == c, "rating"] for c in categories], tick_labels=categories)
axes[1, 1].set_title("Boxplot")
axes[1, 1].tick_params(axis="x", labelrotation=20)
plt.show()


In [ ]:
# ذخیرهٔ نمودار با کیفیت مناسب
# بهتر است قبل از plt.show() ذخیره کنی.
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df, x="revenue", bins=25, ax=ax, color="steelblue")
ax.set_title("Revenue Distribution")
fig.savefig("revenue_distribution.png", dpi=300, bbox_inches="tight")
fig.savefig("revenue_distribution.pdf", bbox_inches="tight")
plt.show()

# بستن شکل‌ها در حلقه‌ها یا اسکریپت‌های طولانی
plt.close(fig)


## 6) شروع کار با Seaborn
Seaborn روی Matplotlib ساخته شده و برای نمودارهای آماری، گروه‌بندی داده و نمایش ارتباط متغیرها دستورهای ساده‌تری دارد. معمولاً بهتر است داده را با آرگومان `data=df` بدهی و ستون‌ها را با نام مشخص کنی.

In [ ]:
# الگوی عمومی
# sns.plot_function(data=df, x="column_x", y="column_y", hue="group_column")

# scatterplot: رابطهٔ دو متغیر، تفکیک رنگی بر اساس دسته
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="ad_spend", y="revenue", hue="category", size="quantity",
                sizes=(25, 150), alpha=0.75)
plt.title("Revenue vs Ad Spend by Category")
plt.tight_layout(); plt.show()

# lineplot: روندها؛ می‌تواند میانگین و بازهٔ عدم‌قطعیت را نشان دهد
plt.figure(figsize=(8, 4))
sns.lineplot(data=df_sales, x="date", y="revenue", hue="category", marker="o", errorbar=None)
plt.title("Monthly Revenue by Category")
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()


## 7) توزیع داده‌ها در Seaborn

In [ ]:
# histplot: هیستوگرام، امکان تفکیک بر اساس hue
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="revenue", bins=25, kde=True, color="teal")
plt.title("Histogram with KDE")
plt.show()

# kdeplot: برآورد چگالی توزیع
plt.figure(figsize=(7, 4))
sns.kdeplot(data=df, x="revenue", fill=True, alpha=0.35, linewidth=2, color="steelblue")
plt.title("Kernel Density Estimate")
plt.show()

# ecdfplot: توزیع تجمعی تجربی
plt.figure(figsize=(7, 4))
sns.ecdfplot(data=df, x="revenue", hue="category")
plt.title("Empirical Cumulative Distribution")
plt.show()

# displot: نمودار توزیع سطح شکل (figure-level)؛ kind می‌تواند hist، kde یا ecdf باشد
sns.displot(data=df, x="revenue", col="category", col_wrap=2, kind="hist", bins=20,
            height=3.2, color="steelblue", facet_kws={"sharex": False, "sharey": False})
plt.show()


## 8) نمودارهای دسته‌ای (Categorical plots)

In [ ]:
# countplot: تعداد رکوردها در هر دسته
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="category", order=df["category"].value_counts().index, color="steelblue")
plt.title("Number of Records by Category")
plt.xticks(rotation=20); plt.show()

# barplot: برآورد آماری یک مقدار عددی برای هر دسته (پیش‌فرض معمولاً میانگین است)
plt.figure(figsize=(7, 4))
sns.barplot(data=df, x="category", y="revenue", errorbar="sd", capsize=0.1, color="steelblue")
plt.title("Mean Revenue by Category (SD error bars)")
plt.xticks(rotation=20); plt.show()

# boxplot: میانه، چارک‌ها و نقاط پرت
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="category", y="revenue", color="steelblue")
plt.title("Revenue by Category")
plt.xticks(rotation=20); plt.show()

# violinplot: شکل توزیع هر گروه
plt.figure(figsize=(7, 4))
sns.violinplot(data=df, x="category", y="rating", inner="quartile", cut=0, color="steelblue")
plt.title("Rating Distribution by Category")
plt.xticks(rotation=20); plt.show()

# stripplot: نشان دادن تک‌تک مشاهدات
plt.figure(figsize=(7, 4))
sns.stripplot(data=df.sample(180, random_state=42), x="category", y="rating",
              jitter=True, alpha=0.6, color="steelblue")
plt.title("Individual Ratings by Category")
plt.xticks(rotation=20); plt.show()

# swarmplot: نمایش نقاط بدون هم‌پوشانی زیاد (برای دیتاست‌های کوچک/متوسط مناسب‌تر است)
plt.figure(figsize=(7, 4))
sns.swarmplot(data=df.sample(120, random_state=42), x="category", y="rating", size=3, color="steelblue")
plt.title("Swarm Plot of Ratings")
plt.xticks(rotation=20); plt.show()

# pointplot: مقایسهٔ مقادیر آماری گروه‌ها با نقاط و خطوط
plt.figure(figsize=(7, 4))
sns.pointplot(data=df, x="category", y="rating", errorbar="sd", capsize=0.1, color="steelblue")
plt.title("Point Plot of Mean Rating")
plt.xticks(rotation=20); plt.show()


## 9) روابط، رگرسیون و همبستگی

In [ ]:
# regplot: پراکندگی به همراه خط رگرسیون
plt.figure(figsize=(7, 4))
sns.regplot(data=df, x="ad_spend", y="revenue", color="steelblue", scatter_kws={"alpha": 0.45},
            line_kws={"color": "red"})
plt.title("Regression Plot")
plt.show()

# lmplot: نمودار رگرسیون با قابلیت تفکیک بر اساس دسته / ایجاد پنل‌ها
sns.lmplot(data=df, x="ad_spend", y="revenue", hue="category", height=4, aspect=1.4,
           scatter_kws={"alpha": 0.35, "s": 20}, ci=95)
plt.show()

# residplot: بررسی باقیمانده‌های مدل خطی
plt.figure(figsize=(7, 4))
sns.residplot(data=df, x="ad_spend", y="revenue", lowess=True,
              scatter_kws={"alpha": 0.4}, line_kws={"color": "red"})
plt.title("Residual Plot")
plt.show()

# محاسبهٔ همبستگی عددی
numeric_cols = ["unit_price", "quantity", "discount", "revenue", "profit", "ad_spend", "rating", "age"]
corr = df[numeric_cols].corr(numeric_only=True)
print(corr.round(2))

# Heatmap: مقادیر همبستگی
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, linewidths=0.5, square=True)
plt.title("Correlation Matrix")
plt.tight_layout(); plt.show()


## 10) نمودارهای چندمتغیره و چندبخشی در Seaborn

In [ ]:
# pairplot: روابط جفتی میان چند متغیر عددی؛ برای بررسی اولیهٔ داده مفید است
sns.pairplot(df.sample(180, random_state=42), vars=["revenue", "profit", "ad_spend", "rating"],
             hue="category", corner=True, plot_kws={"alpha": 0.5, "s": 18})
plt.show()

# jointplot: رابطهٔ دو متغیر به همراه توزیع‌های حاشیه‌ای
sns.jointplot(data=df, x="ad_spend", y="revenue", kind="reg", height=6,
              scatter_kws={"alpha": 0.4, "s": 20})
plt.show()

# relplot: نمودار رابطه‌ای در چند پنل (figure-level)
sns.relplot(data=df, x="ad_spend", y="revenue", hue="category", col="city",
            col_wrap=3, kind="scatter", height=2.7, facet_kws={"sharex": True, "sharey": False})
plt.show()

# catplot: نمودار دسته‌ای در چند پنل؛ kind می‌تواند bar، box، violin، strip و ... باشد
sns.catplot(data=df, x="category", y="rating", col="city", col_wrap=3,
            kind="box", height=2.7, sharey=True)
plt.show()


## 11) تنظیم ظاهر و رنگ‌ها در Seaborn / Matplotlib

In [ ]:
# theme و context کلی
sns.set_theme(style="whitegrid", context="talk", palette="muted")
# style های رایج: whitegrid, darkgrid, white, dark, ticks
# context های رایج: paper, notebook, talk, poster

# مشاهدهٔ پالت‌های رنگی
sns.palplot(sns.color_palette("deep"))
plt.show()
sns.palplot(sns.color_palette("viridis", 8))
plt.show()

# پالت‌های پراستفاده: deep, muted, pastel, bright, dark, colorblind,
# viridis, magma, rocket, cubehelix, coolwarm

# کنترل رنگ، اندازه و شفافیت
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="unit_price", y="profit", hue="category",
                palette="Set2", size="quantity", sizes=(20, 130), alpha=0.7)
plt.title("Custom Palette and Sizes")
plt.tight_layout(); plt.show()

# بازگرداندن ظاهر پیش‌فرض مناسب برای بقیهٔ نمودارها
sns.set_theme(style="whitegrid", context="notebook", palette="deep")


## 12) نکات پیشرفته و کاربردی

In [ ]:
# مرتب‌سازی ترتیب دسته‌ها به شکل دلخواه
order = ["Electronics", "Home", "Sports", "Clothing"]
plt.figure(figsize=(7, 4))
sns.barplot(data=df, x="category", y="profit", order=order, errorbar=None, color="steelblue")
plt.title("Profit by a Custom Category Order")
plt.show()

# محور لگاریتمی، وقتی توزیع بسیار چوله است (فقط برای مقادیر مثبت)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
sns.histplot(data=df, x="revenue", ax=axes[0], bins=25, color="steelblue")
axes[0].set_title("Original Scale")
sns.histplot(data=df, x="revenue", ax=axes[1], bins=25, color="steelblue")
axes[1].set_xscale("log")
axes[1].set_title("Log X Scale")
plt.show()

# درصد یا نسبت با pandas آماده و بعد رسم با Matplotlib
share = df["category"].value_counts(normalize=True).mul(100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(share.index, share.values, color=sns.color_palette("deep", len(share)))
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set(title="Share of Records by Category", ylabel="Percent", xlabel="Category")
ax.tick_params(axis="x", labelrotation=20)
plt.tight_layout(); plt.show()


## 13) مرجع سریع توابع مهم

### Matplotlib

| هدف | تابع / دستور | کاربرد |
|---|---|---|
| شکل و محور | `plt.figure()`, `plt.subplots()` | ساخت شکل و محور |
| خطی | `ax.plot(x, y)` | روند یا سری زمانی |
| پراکندگی | `ax.scatter(x, y)` | رابطهٔ دو متغیر |
| میله‌ای | `ax.bar(x, y)`, `ax.barh(y, x)` | مقایسهٔ دسته‌ها |
| هیستوگرام | `ax.hist(x, bins=...)` | توزیع مقدارها |
| جعبه‌ای | `ax.boxplot(data)` | میانه، چارک و نقاط پرت |
| دایره‌ای | `ax.pie(values, labels=...)` | سهم دسته‌ها با تعداد کم |
| خطای اندازه‌گیری | `ax.errorbar(x, y, yerr=...)` | نمایش عدم‌قطعیت |
| خط مرجع | `ax.axhline()`, `ax.axvline()` | میانگین/میانه یا آستانه |
| ناحیهٔ سایه‌دار | `ax.fill_between()` | بازه یا عدم‌قطعیت |
| عنوان و برچسب | `ax.set_title()`, `ax.set_xlabel()`, `ax.set_ylabel()` | نام‌گذاری |
| محدودهٔ محور | `ax.set_xlim()`, `ax.set_ylim()` | تنظیم محدوده |
| راهنما | `ax.legend()` | نمایش labelها |
| شبکه | `ax.grid()` | خطوط راهنمای محور |
| توضیح نقطه | `ax.annotate()`, `ax.text()` | نوشتن متن روی شکل |
| چیدمان | `fig.tight_layout()`, `layout="constrained"` | جلوگیری از هم‌پوشانی |
| ذخیره | `fig.savefig("plot.png", dpi=300, bbox_inches="tight")` | ذخیرهٔ تصویر با کیفیت |
| بستن شکل | `plt.close(fig)` | آزادسازی منابع |

### Seaborn

| هدف | تابع | کاربرد |
|---|---|---|
| پراکندگی | `sns.scatterplot()` | رابطهٔ دو متغیر و تفکیک با `hue` |
| روند | `sns.lineplot()` | روند و خلاصه‌سازی داده |
| هیستوگرام | `sns.histplot()` | توزیع و در صورت نیاز KDE |
| چگالی | `sns.kdeplot()` | نمایش نرم توزیع |
| تجمعی | `sns.ecdfplot()` | توزیع تجمعی تجربی |
| توزیع چندپنلی | `sns.displot()` | هیستوگرام/KDE/ECDF در پنل‌ها |
| شمارش دسته‌ها | `sns.countplot()` | شمارش ردیف‌های هر دسته |
| میانگین دسته‌ها | `sns.barplot()` | برآورد آماری و بازهٔ خطا |
| جعبه‌ای | `sns.boxplot()` | پراکندگی و نقاط پرت |
| ویولن | `sns.violinplot()` | شکل توزیع در گروه‌ها |
| نقاط دسته‌ای | `sns.stripplot()`, `sns.swarmplot()` | نمایش مشاهدات منفرد |
| رگرسیون | `sns.regplot()`, `sns.lmplot()` | رابطه و خط برازش |
| باقیمانده | `sns.residplot()` | تشخیص الگو در خطاهای مدل |
| همبستگی | `sns.heatmap()` | نمایش ماتریس به شکل رنگی |
| روابط جفتی | `sns.pairplot()` | چندین رابطه در یک خروجی |
| رابطه + توزیع حاشیه‌ای | `sns.jointplot()` | تحلیل دو متغیر |
| روابط چندپنلی | `sns.relplot()` | پراکندگی/خطی در چند پنل |
| دسته‌های چندپنلی | `sns.catplot()` | نمودار دسته‌ای در چند پنل |
| ظاهر کلی | `sns.set_theme()` | تنظیم تم و ظاهر |
| رنگ‌ها | `sns.color_palette()` | انتخاب پالت رنگی |

### آرگومان‌های رایج در Seaborn

- `data=df`: دیتافریم منبع
- `x="column"`, `y="column"`: ستون‌های محور افقی و عمودی
- `hue="group"`: تفکیک گروه‌ها با رنگ
- `style="group"`: تفکیک با نوع نشانگر/خط در نمودارهای پشتیبانی‌شده
- `size="column"`: اندازهٔ نقاط بر اساس متغیر
- `col="group"`, `row="group"`: ساخت پنل برای دسته‌ها
- `palette="viridis"`: پالت رنگ
- `order=[...]`: ترتیب دسته‌ها
- `errorbar="sd"` یا `errorbar=None`: تنظیم بازهٔ خطا در توابع پشتیبانی‌شده
- `alpha=0.5`: شفافیت نقاط
- `figsize=(width, height)`: اندازهٔ شکل در Matplotlib
- `bins=20`: تعداد بازه‌های هیستوگرام


## 14) کدام نمودار را انتخاب کنم؟

| سؤال / هدف | نمودار پیشنهادی | تابع نمونه |
|---|---|---|
| روند در طول زمان چیست؟ | Line plot | `sns.lineplot()` / `ax.plot()` |
| دو متغیر چه رابطه‌ای دارند؟ | Scatter plot | `sns.scatterplot()` / `ax.scatter()` |
| توزیع یک متغیر چگونه است؟ | Histogram یا KDE | `sns.histplot()` / `sns.kdeplot()` |
| گروه‌ها از نظر مقدار عددی چه تفاوتی دارند؟ | Box/Violin plot | `sns.boxplot()` / `sns.violinplot()` |
| تعداد هر گروه چقدر است؟ | Count plot | `sns.countplot()` |
| میانگین دسته‌ها چقدر است؟ | Bar plot | `sns.barplot()` |
| رابطهٔ خطی تقریبی وجود دارد؟ | Regression plot | `sns.regplot()` |
| روابط چندین متغیر عددی چیست؟ | Pair plot / Heatmap | `sns.pairplot()` / `sns.heatmap()` |
| ترکیب چند گروه یا شهر را مقایسه کنم؟ | Facet plot | `sns.relplot()` / `sns.catplot()` |

**نکات مهم:**
- همبستگی الزاماً به معنای رابطهٔ علت‌ومعلولی نیست.
- در `barplot`، هر ستون معمولاً میانگین متغیر عددی را نشان می‌دهد، نه تعداد رکوردها؛ برای تعداد از `countplot` استفاده کن.
- برای داده‌های بسیار پرتعداد، `swarmplot` و `pairplot` ممکن است کند یا شلوغ شوند.
- از نمودار دایره‌ای برای دسته‌های زیاد استفاده نکن؛ bar chart معمولاً مقایسه را آسان‌تر می‌کند.
- عنوان، واحد محور، منبع داده و توضیح مناسب را فراموش نکن.
- نمودار را برای پاسخ‌دادن به یک سؤال مشخص انتخاب کن، نه فقط برای زیباتر شدن گزارش.


## 15) الگوی تمیز برای استفاده در پروژه‌های واقعی

In [ ]:
# الگوی پیشنهادی: تابع مستقل برای رسم و ذخیرهٔ یک نمودار

def plot_category_revenue(data: pd.DataFrame, output_path: str | None = None):
    """Plot mean revenue by category and optionally save the figure."""
    summary = data.groupby("category", as_index=False)["revenue"].mean()
    summary = summary.sort_values("revenue", ascending=False)

    fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
    sns.barplot(data=summary, x="category", y="revenue", color="steelblue", ax=ax, errorbar=None)
    ax.set_title("Average Revenue by Category")
    ax.set_xlabel("Category")
    ax.set_ylabel("Average revenue")
    ax.tick_params(axis="x", labelrotation=20)
    ax.grid(axis="y", alpha=0.25)

    if output_path:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
    return fig, ax

fig, ax = plot_category_revenue(df, output_path=None)
plt.show()


---

### پایان
این فایل برای یادگیری و مراجعهٔ سریع آماده شده است. می‌توانی آن را در Jupyter Notebook، JupyterLab یا Google Colab باز کنی و سلول‌ها را یکی‌یکی اجرا کنی. برای یک پروژهٔ واقعی، دادهٔ نمونهٔ سلول دوم را با دیتاست خودت جایگزین کن و ستون‌های `x`، `y` و `hue` را متناسب با سؤال تحلیلی‌ات تغییر بده.
